# Notebook 12: Fine-tuning GPT-2 for Text Generation (SOLUTION)

**Module 4 · Text Generation**  
*Author: Axel Sirota, Data Trainers LLC*

## The Scenario

In Notebook 11 you trained an LSTM language model on Airbnb rental descriptions. It works, but beyond roughly 30 tokens the output drifts. The reason is structural: the LSTM hidden state is a fixed-size bottleneck, so token 50's memory of token 1 has been squeezed through 49 weight matrices and is severely diluted. Meanwhile you saw BERT outperform the MLP on classification back in Notebook 9. The natural next question: can a transformer *decoder* beat the LSTM at *generation*?

The plan is straightforward. Fine-tune `gpt2` (124M parameters) on the exact same Airbnb dataset you used in Notebook 11, using `AutoModelForCausalLM` and `AutoTokenizer` from HuggingFace together with the standard `Trainer` API for supervised fine-tuning. Then generate text with `model.generate()` using greedy, beam, top-k, top-p, and temperature decoding, and compare output quality, coherence, perplexity, and model size against the LSTM baseline. By the end you will have a fine-tuned GPT-2 that produces coherent 100+ token rental descriptions and a reusable template for fine-tuning any causal LM.

## Learning objectives

By the end of this notebook you will be able to:

1. Contrast encoder-only (BERT) vs. decoder-only (GPT-2) transformers and explain when to use each.
2. Explain causal attention masking: why decoder models only attend to previous tokens.
3. Use `AutoModelForCausalLM` + `AutoTokenizer` to load and tokenize with GPT-2.
4. Prepare causal-LM training data (labels = input_ids shifted by 1, done automatically by HF).
5. Fine-tune GPT-2 with `Trainer` using `DataCollatorForLanguageModeling(mlm=False)`.
6. Generate text with `model.generate(...)` using greedy, beam search, top-k, top-p, and temperature.
7. Compute and compare perplexity for GPT-2 vs. LSTM on held-out Airbnb data.
8. Use `pipeline('text-generation', ...)` for zero-effort inference at deployment.
9. Know the trade-offs: GPT-2 needs 124M params + GPU; LSTM needs ~3M and runs anywhere.

## Prerequisites

- Notebook 11 (LSTM Language Model): you understand the generation task and dataset.
- Notebook 9 (BERT fine-tuning via `Trainer`): you know the HuggingFace training API.
- Comfortable with PyTorch `nn.Module`, `DataLoader`, and `torch.no_grad()`.

## Runtime

Recommended: **Colab with GPU** (T4 is sufficient). Full fine-tuning takes about 45 min on T4. Without GPU, training will be prohibitively slow, so reduce `NUM_EPOCHS` to 1 and `TRAIN_LIMIT` to 2000.

## Section 0. Environment Setup

Install HuggingFace Transformers, Datasets, Accelerate, and Evaluate. On a fresh Colab session this takes about 3 minutes the first time. The model download (124 MB for `gpt2`) happens the first time you call `from_pretrained`.

> **GPU check:** Click *Runtime -> Change runtime type -> T4 GPU* before running this notebook. Training on CPU is roughly 20x slower.

In [ ]:
# Install required packages (run this first in Google Colab).
# If running locally with your virtual environment, you may skip this cell.
!pip install -q \
    "transformers>=4.40" \
    "datasets>=2.19" \
    "accelerate>=0.28" \
    "evaluate>=0.4" \
    "torch>=2.1" \
    "pandas>=2.0" \
    "matplotlib>=3.7"
print("Installation complete.")

In [ ]:
# Imports (visualization, data, model, training).
import os
import math
import time
import random
import warnings
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline,
)
import datasets as hf_datasets

warnings.filterwarnings("ignore")

# --- Reproducibility: seed every RNG we touch ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Device detection ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version      : {torch.__version__}")
print(f"Transformers version : {transformers.__version__}")
print(f"Using device         : {device}")
if device.type == 'cuda':
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nEnvironment ready.")

In [ ]:
# Hyperparameters (defined once, reused everywhere).

SEED           = 42
MODEL_NAME     = 'gpt2'
MAX_LENGTH     = 128
NUM_EPOCHS     = 3
BATCH_SIZE     = 4
GRAD_ACCUM     = 4            # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR             = 5e-5
WARMUP_STEPS   = 100
FP16           = torch.cuda.is_available()
TRAIN_LIMIT    = 5000
TEST_LIMIT     = 500

GEN_MAX_NEW    = 40
GEN_TEMP       = 0.8
GEN_TOP_K      = 50
GEN_TOP_P      = 0.9

print(f"Model           : {MODEL_NAME}")
print(f"Max length      : {MAX_LENGTH} tokens")
print(f"Epochs          : {NUM_EPOCHS}")
print(f"Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"FP16            : {FP16}")

## What We Build Here

Imagine you are a product engineer at Airbnb. A new host uploads a property but writes only three words in the description box: *"Cozy downtown loft"*. The product spec asks for a full 100-word description for review before publication.

The NLP team already tried an LSTM (Notebook 11); it works but drifts after ~30 tokens. Can a transformer do better?

**Pipeline for this notebook:**

```
Raw Airbnb descriptions
        |
        v
  GPT-2 tokenizer (BPE, 50K vocab)
        |
        v
  HF Dataset + DataCollator
        |
        v
  Trainer.train()  <- fine-tune GPT-2 (124M)
        |
        v
  model.generate()  <- greedy / beam / top-k / top-p
        |
        v
  Perplexity comparison: GPT-2 vs LSTM
```

## Section 1. Why Transformers for Generation?

### The LSTM ceiling

The LSTM carries meaning from step to step via a hidden state vector of fixed size. Every token has to "fit" through that bottleneck. By step 50, signal from step 1 has been multiplied by 49 weight matrices, so it is severely attenuated.

### The transformer decoder solution

A transformer decoder (GPT-2) replaces the hidden state with **self-attention**: at every step, it attends directly to every previous token, with no bottleneck and full memory of the past. **Causal masking** prevents looking at future tokens, keeping the model autoregressive.

### GPT-2 model family

| Model | Parameters | Size | Notes |
|---|---|---|---|
| `gpt2` (small) | 124M | ~500MB | We use this. Fast to fine-tune on Colab. |
| `gpt2-medium` | 355M | ~1.4GB | Higher quality, needs more VRAM. |
| `gpt2-large` | 774M | ~3GB | Best quality in the family. |
| `gpt2-xl` | 1.5B | ~6GB | Rarely fits on free Colab. |

## Section 2. Using Pretrained GPT-2 Out of the Box

In [ ]:
# Load GPT-2 tokenizer.
# IMPORTANT: GPT-2 has no padding token by default, so we must set one.
print(f"Loading tokenizer for '{MODEL_NAME}' ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Without this line, batched tokenization will crash.
tokenizer.pad_token = tokenizer.eos_token  # <-- mandatory GPT-2 workaround

print(f"Vocab size        : {tokenizer.vocab_size:,}")
print(f"Max model length  : {tokenizer.model_max_length}")
print(f"EOS token         : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"PAD token         : {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})")
print(f"BOS token         : {tokenizer.bos_token!r}  (id={tokenizer.bos_token_id})")

In [ ]:
# See how GPT-2 tokenizes a sample rental description.
sample_text = "This cozy downtown loft features exposed brick and modern amenities."
tokens = tokenizer.tokenize(sample_text)
ids    = tokenizer.encode(sample_text)

print(f"Original text  : {sample_text}")
print(f"Tokens ({len(tokens)})     : {tokens}")
print(f"Token IDs      : {ids}")
print()
print("Note: BPE splits words into subword pieces, so no OOV tokens ever.")

In [ ]:
# Load base GPT-2 model (no fine-tuning yet).
print(f"Loading '{MODEL_NAME}' model (~500MB, may take a moment) ...")
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model = base_model.to(device)
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Parameters    : {n_params:,}  (~{n_params/1e6:.0f}M)")
print(f"Device        : {next(base_model.parameters()).device}")

In [ ]:
# Quick demo: pretrained GPT-2 on a generic prompt (before fine-tuning).
generic_pipe = pipeline(
    'text-generation',
    model=base_model,
    tokenizer=tokenizer,
    device=0 if device.type == 'cuda' else -1,
)

prompt_generic = "The capital of France is"
out = generic_pipe(
    prompt_generic,
    max_new_tokens=30,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)[0]['generated_text']
print(f"Prompt  : {prompt_generic!r}")
print(f"Output  : {out}")

In [ ]:
# Now try an Airbnb-style prompt.
# Pretrained GPT-2 has never seen Airbnb listings, so output will be generic.
prompt_airbnb = "This rental offers"
out = generic_pipe(
    prompt_airbnb,
    max_new_tokens=40,
    do_sample=True,
    temperature=GEN_TEMP,
    top_k=GEN_TOP_K,
    top_p=GEN_TOP_P,
    pad_token_id=tokenizer.eos_token_id,
)[0]['generated_text']
print(f"Prompt  : {prompt_airbnb!r}")
print(f"Output  : {out}")
print()
print("Output is grammatically OK but NOT Airbnb-style. Fine-tuning will fix this.")

In [ ]:
# Helper: tokenize a prompt and generate with model.generate().

def generate_text(model, prompt, max_new_tokens=GEN_MAX_NEW, **gen_kwargs):
    """Tokenize `prompt`, generate tokens, decode and return the full string."""
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **gen_kwargs,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Five decoding strategies on the same prompt
prompt = "This rental offers stunning views of"

strategies = [
    ("Greedy",          dict(do_sample=False)),
    ("Beam search (4)", dict(do_sample=False, num_beams=4, early_stopping=True)),
    ("Temp=0.5",        dict(do_sample=True, temperature=0.5)),
    ("Top-K=50",        dict(do_sample=True, top_k=50)),
    ("Top-P=0.9",       dict(do_sample=True, top_p=0.9, top_k=0)),
]

print(f"Prompt: {prompt!r}\n")
for name, kwargs in strategies:
    text = generate_text(base_model, prompt, **kwargs)
    continuation = text[len(prompt):].strip()
    print(f"[{name:<17}] {continuation[:120]}")

### Lab 1. Explore Decoding Strategies

Try the base model with different decoding strategies on a prompt of your choice.

In [ ]:
# Solution. Lab 1: decoding strategies exploration.

# 1. Choose a prompt
lab_prompt = "The apartment is located"  # Good Airbnb-style prompt

# 2a. Greedy: always picks the single most probable token
#     -> deterministic, often repetitive ("the the the...")
greedy_output = generate_text(base_model, lab_prompt, do_sample=False)

# 2b. Temperature=1.0: sample from the raw softmax distribution
#     -> most random/diverse, can be incoherent
temp_output = generate_text(base_model, lab_prompt, do_sample=True, temperature=1.0)

# 2c. Top-P=0.95 nucleus sampling: keep smallest set of tokens with cumprob >= 0.95
#     -> adaptive diversity, often the best balance
# top_k=0 disables top-k so ONLY top-p filtering applies
topp_output = generate_text(base_model, lab_prompt, do_sample=True, top_p=0.95, top_k=0)

# 3. Print results
print(f"Prompt : {lab_prompt!r}\n")
print(f"[Greedy]    : {greedy_output[len(lab_prompt):].strip()[:150]}")
print(f"[Temp=1.0]  : {temp_output[len(lab_prompt):].strip()[:150]}")
print(f"[Top-P=0.95]: {topp_output[len(lab_prompt):].strip()[:150]}")

# Discussion:
# Greedy: deterministic, tends toward safe/common phrases, can loop ("the the the")
# Temp=1.0: most diverse but also most likely to produce incoherent text
# Top-P=0.95: usually the best trade-off, coherent AND varied.
# The right strategy depends on the task:
# for Airbnb descriptions we want diversity plus domain relevance, so top-p + moderate temperature.

## Section 3. Data Preparation for Causal LM Fine-tuning

For a causal LM, each training sample is a sequence of tokens. HuggingFace's `DataCollatorForLanguageModeling(mlm=False)` automatically constructs `labels = input_ids` shifted by 1. We prefix each description with `<|endoftext|>` (GPT-2's BOS/EOS).

In [ ]:
# Download the Airbnb descriptions dataset (same train/test split as Notebook 11).
TRAIN_URL = "https://www.dropbox.com/scl/fi/rbrynlq7871cshi0krftj/train_corpus_descriptions_airbnb.csv?rlkey=td1pfjgqjccap0xu9g4eliube&dl=1"
TEST_URL  = "https://www.dropbox.com/scl/fi/eys05bzwwnhskadqh7aux/test_corpus_descriptions_airbnb.csv?rlkey=p1zuz90khh5t7dx3hkfba1dzm&dl=1"

TRAIN_PATH = "./airbnb_train.csv"
TEST_PATH  = "./airbnb_test.csv"

if not os.path.exists(TRAIN_PATH):
    print("Downloading Airbnb train set ...")
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)

if not os.path.exists(TEST_PATH):
    print("Downloading Airbnb test set ...")
    urllib.request.urlretrieve(TEST_URL, TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

text_col = [c for c in train_df.columns if 'descrip' in c.lower() or 'text' in c.lower()][0]
print(f"Text column: {text_col!r}")

train_texts = train_df[text_col].dropna().astype(str).str.strip().tolist()
test_texts  = test_df[text_col].dropna().astype(str).str.strip().tolist()

if TRAIN_LIMIT:
    random.shuffle(train_texts)
    train_texts = train_texts[:TRAIN_LIMIT]
if TEST_LIMIT:
    test_texts = test_texts[:TEST_LIMIT]

print(f"\nTrain descriptions : {len(train_texts):,}")
print(f"Test descriptions  : {len(test_texts):,}")
print(f"\nSample description:\n{train_texts[0][:300]}")

In [ ]:
# Format each description as:  <|endoftext|> + description
BOS = tokenizer.bos_token  # '<|endoftext|>'

train_formatted = [f"{BOS}{text}" for text in train_texts]
test_formatted  = [f"{BOS}{text}" for text in test_texts]

print(f"BOS token: {BOS!r}")
print(f"Formatted train[0]:\n{train_formatted[0][:200]}\n")

# Token length analysis
sample_lens = [
    len(tokenizer.encode(t, truncation=False)) for t in train_formatted[:500]
]
print(f"Token length stats (first 500 samples):")
print(f"  mean   = {np.mean(sample_lens):.0f}")
print(f"  median = {np.median(sample_lens):.0f}")
print(f"  p90    = {np.percentile(sample_lens, 90):.0f}")
print(f"  max    = {max(sample_lens)}")
print(f"  MAX_LENGTH = {MAX_LENGTH}  (truncates ~{sum(l > MAX_LENGTH for l in sample_lens)/len(sample_lens)*100:.0f}%)")

In [ ]:
# Tokenize all texts and convert to HuggingFace Dataset objects.

def tokenize_batch(texts, max_length=MAX_LENGTH):
    """Tokenize a list of texts and return a HuggingFace Dataset."""
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_overflowing_tokens=False,
    )
    data = {'input_ids': encodings['input_ids'], 'attention_mask': encodings['attention_mask']}
    return hf_datasets.Dataset.from_dict(data)

print("Tokenizing train set ...")
train_dataset = tokenize_batch(train_formatted)
print("Tokenizing test set ...")
test_dataset  = tokenize_batch(test_formatted)

print(f"\nTrain dataset : {train_dataset}")
print(f"Test dataset  : {test_dataset}")
print(f"\nFirst sample token IDs: {train_dataset[0]['input_ids'][:20]} ...")

In [ ]:
# DataCollator for causal LM.
# mlm=False  ->  don't randomly mask tokens (that's BERT)
# It pads, sets labels = input_ids.clone(), and marks padding positions with -100.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # CRITICAL: False = causal LM (GPT-2); True = masked LM (BERT)
)

# Sanity check
sample_batch = [train_dataset[i] for i in range(2)]
collated = data_collator(sample_batch)
print("DataCollator output for 2 samples:")
for k, v in collated.items():
    print(f"  {k:<20} shape={tuple(v.shape)}  dtype={v.dtype}")
print()
print("labels=-100 at padding positions (ignored by loss):")
print(collated['labels'][0][:30])

### Lab 2. Prepare the Dataset Pipeline

In [ ]:
# Solution. Lab 2: dataset pipeline from scratch.

# 1. Take 20 descriptions and format with BOS
# We prepend BOS so the model learns to start descriptions from this special token.
mini_texts = [f"{BOS}{text}" for text in train_texts[:20]]

# 2. Tokenize with truncation at 64 tokens (shorter for this demo)
mini_dataset = tokenize_batch(mini_texts, max_length=64)

# 3. Create collator with mlm=False (causal LM, NOT masked LM)
mini_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# 4. Collate all 20 samples into a single batch
# DataCollator pads shorter sequences to the length of the longest in the batch.
mini_batch = mini_collator([mini_dataset[i] for i in range(20)])

# Print shapes
print("Batch shapes:")
for k, v in mini_batch.items():
    print(f"  {k:<20} shape={tuple(v.shape)}")

# 5. Verify -100 on padding positions
# -100 is PyTorch's CrossEntropyLoss ignore_index, so padded positions don't affect loss.
mask   = mini_batch['attention_mask']
labels = mini_batch['labels']
n_ignored = (labels == -100).sum().item()
n_pad     = (mask == 0).sum().item()
print(f"\nPadding positions (mask==0)      : {n_pad}")
print(f"Ignored label positions (-100)   : {n_ignored}")
# n_ignored >= n_pad because DataCollator also sets the FIRST token's label to -100
# in some versions. The important thing: all pad positions ARE -100.
all_pad_ignored = (labels[mask == 0] == -100).all().item()
print(f"All padding positions are -100   : {all_pad_ignored}")

# Common mistake: using mlm=True with GPT-2 would randomly mask input tokens
# and try to predict the masked tokens (BERT-style), which is wrong for a
# decoder-only left-to-right model.

## Section 4. Fine-tuning with the HuggingFace Trainer

In [ ]:
# Load a fresh copy of GPT-2 for fine-tuning.
print(f"Loading '{MODEL_NAME}' for fine-tuning ...")
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
ft_model = ft_model.to(device)
ft_model.resize_token_embeddings(len(tokenizer))
print(f"Model ready. Parameters: {sum(p.numel() for p in ft_model.parameters()):,}")

In [ ]:
# TrainingArguments.
training_args = TrainingArguments(
    output_dir='./gpt2-airbnb',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    fp16=FP16,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=SEED,
)

print("TrainingArguments configured.")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  FP16                 : {FP16}")

In [ ]:
# Build the Trainer and start training.
trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting fine-tuning ...")
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} minutes.")
print(f"Final training loss : {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on held-out test set and compute perplexity.
eval_results = trainer.evaluate()
eval_loss    = eval_results['eval_loss']
perplexity   = math.exp(eval_loss)

print(f"Test eval_loss  : {eval_loss:.4f}")
print(f"Test perplexity : {perplexity:.2f}")
print()
print("Reference: Notebook 11 LSTM perplexity ~100 (word-level)")
print("Expected GPT-2 fine-tuned perplexity    : ~20-40 (BPE, different tokenizer!)")
print()
print("IMPORTANT CAVEAT: NOT directly comparable (different tokenizers).")
print("Different vocabularies = different token counts = incomparable perplexity.")

In [ ]:
# Save the fine-tuned model to disk.
SAVE_DIR = './gpt2-airbnb-final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Fine-tuned model saved to: {SAVE_DIR}")

# Reload and verify
reloaded = AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(device)
reloaded.eval()
ft_model.eval()

test_prompt = "This spacious apartment features"
out_orig   = generate_text(ft_model, test_prompt, do_sample=False)
out_reload = generate_text(reloaded, test_prompt, do_sample=False)
print(f"Original  : {out_orig[:150]}")
print(f"Reloaded  : {out_reload[:150]}")
print(f"Match     : {out_orig == out_reload}")

### Lab 3. Retrain with Fewer Epochs

In [ ]:
# Solution. Lab 3: 1-epoch vs 3-epoch comparison.

# 1. TrainingArguments for 1 epoch, same everything else
args_1ep = TrainingArguments(
    output_dir='./gpt2-airbnb-1ep',
    num_train_epochs=1,            # <-- only difference
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    fp16=FP16,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',
    seed=SEED,
)

# 2. Fresh model: ALWAYS start from pretrained weights, not from ft_model
# (otherwise we'd be continuing training from epoch 3, not starting fresh)
model_1ep = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_1ep.resize_token_embeddings(len(tokenizer))

# 3. Trainer
trainer_1ep = Trainer(
    model=model_1ep,
    args=args_1ep,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# 4. Train + evaluate
print("Training 1-epoch model ...")
trainer_1ep.train()
res_1ep = trainer_1ep.evaluate()
ppl_1ep = math.exp(res_1ep['eval_loss'])
print(f"\n1-epoch perplexity : {ppl_1ep:.2f}")
print(f"3-epoch perplexity : {perplexity:.2f}")
print(f"Improvement        : {(ppl_1ep - perplexity)/ppl_1ep*100:.1f}% lower ppl with 3 epochs")

# 5. Side-by-side samples
model_1ep.eval()
cmp_prompt = "The bedroom has"
print(f"\nPrompt: {cmp_prompt!r}")
for m, label in [(model_1ep, '1-epoch'), (ft_model, '3-epoch')]:
    text = generate_text(m, cmp_prompt, do_sample=True, temperature=0.8, top_k=50)
    print(f"[{label}] {text[len(cmp_prompt):].strip()[:150]}")

# Discussion:
# 1-epoch: learns the domain vocabulary but may still produce slightly generic sentences.
# 3-epoch: more Airbnb-specific phrasing, better sentence structure.
# Beyond 5 epochs on small data: risk of overfitting (loss decreases, but outputs
# start to regurgitate training examples verbatim).

## Section 5. Decoding Strategies Deep Dive

### Greedy and Beam search

**Greedy**: `next_token = argmax(logits)`. Deterministic, simple, but repetitive.  
**Beam search**: maintain top-B sequences, choose highest cumulative log-prob. Less repetitive than greedy but still deterministic and "safe".

### Temperature, Top-K, Top-P

**Temperature**: divide logits by `T` before softmax. Lower T = sharper distribution (less creative), higher T = flatter (more creative/chaotic).  
**Top-K**: zero out all but top-K logits. Prevents sampling garbage tokens.  
**Top-P (nucleus)**: keep smallest set of tokens with cumulative probability >= P. Adaptive, usually the best default.  
**Repetition penalty**: divide logits of already-generated tokens by `rep_penalty > 1`. Suppresses loops.

In [ ]:
# Five decoding strategies on the FINE-TUNED model, all five Airbnb prompts.
ft_model.eval()

test_prompts = [
    "This rental offers stunning views of",
    "The kitchen features",
    "Walking distance to",
    "Perfect for couples,",
    "The neighborhood is",
]

strategies_ft = [
    ("Greedy",              dict(do_sample=False)),
    ("Beam (k=4)",          dict(do_sample=False, num_beams=4, early_stopping=True)),
    ("Temp=0.8",            dict(do_sample=True, temperature=0.8)),
    ("Top-K=50",            dict(do_sample=True, top_k=50)),
    ("Top-P=0.9, T=0.8",    dict(do_sample=True, top_p=0.9, top_k=0, temperature=0.8)),
]

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt!r}")
    print('='*70)
    for name, kwargs in strategies_ft:
        text = generate_text(ft_model, prompt, **kwargs)
        cont = text[len(prompt):].strip()[:100]
        print(f"  [{name:<20}] {cont}")

### Lab 4. Your Best Airbnb Description

In [ ]:
# Solution. Lab 4: best Airbnb description.

# 1. Prompt with a specific scenario
my_prompt = "Charming beach cottage with"

# 2a. Strategy A: temperature=0.7, top-k=40. Balanced quality and diversity.
output_a = generate_text(
    ft_model, my_prompt,
    do_sample=True, temperature=0.7, top_k=40, max_new_tokens=60
)

# 2b. Strategy B: top-p=0.92, temperature=0.9. Nucleus sampling, slightly more creative.
output_b = generate_text(
    ft_model, my_prompt,
    do_sample=True, top_p=0.92, top_k=0, temperature=0.9, max_new_tokens=60
)

# 2c. Strategy C: repetition_penalty=1.2. Discourages repeating the same phrases.
output_c = generate_text(
    ft_model, my_prompt,
    do_sample=True, top_p=0.9, temperature=0.8, repetition_penalty=1.2, max_new_tokens=60
)

# 3. Print all outputs
print(f"Prompt: {my_prompt!r}\n")
for label, out in [("Strategy A (temp=0.7, k=40)", output_a),
                   ("Strategy B (top-p=0.92)",     output_b),
                   ("Strategy C (rep_pen=1.2)",     output_c)]:
    print(f"--- {label} ---")
    print(out)
    print()

# 4. My favorite is Strategy C with repetition_penalty=1.2.
# Because: repetition_penalty prevents the model from looping on phrases like
# 'the apartment is' or 'the kitchen has'. The result reads more like a real
# Airbnb listing with varied sentence structure.
# For production: combine top_p=0.9 + temperature=0.8 + repetition_penalty=1.1-1.2.

## Section 6. Head-to-Head: GPT-2 vs. LSTM

In [ ]:
# Reference LSTM outputs (from a completed Notebook 11 run).
lstm_reference_outputs = {
    "This rental offers stunning views of": 
        "the city skyline from the private balcony . the apartment is located in the heart of . "
        "the apartment is a great place to stay in . the apartment is clean and ",
    "The kitchen features":
        "a fully equipped kitchen with stainless steel appliances . the kitchen is equipped with a "
        "full kitchen with all the amenities you need . the kitchen is perfect for",
    "Walking distance to":
        "the beach and local restaurants . the apartment is located in a quiet neighborhood . "
        "the apartment is a short walk from the subway . the apartment is a",
}

comparison_prompts = list(lstm_reference_outputs.keys())
print("=" * 70)
print("HEAD-TO-HEAD: GPT-2 fine-tuned vs LSTM reference outputs")
print("=" * 70)

for prompt in comparison_prompts:
    gpt2_out  = generate_text(ft_model, prompt, do_sample=True, temperature=0.8, top_k=50, top_p=0.9)
    gpt2_cont = gpt2_out[len(prompt):].strip()[:200]
    lstm_cont = lstm_reference_outputs[prompt]

    print(f"\nPROMPT: {prompt!r}")
    print(f"  LSTM   : {lstm_cont[:150]}")
    print(f"  GPT-2  : {gpt2_cont[:150]}")
    print()
    # Key qualitative differences you should see:
    # LSTM: short sentences, repetitive structure ("the apartment is... the apartment is...")
    # GPT-2: longer, more varied sentence structure, more domain-appropriate vocabulary
    # This is the practical payoff of 124M params + pretraining on web text.

In [ ]:
# Quantitative comparison table.
gpt2_size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(SAVE_DIR) if os.path.exists(SAVE_DIR)
    for f in files
) / 1e6

print("=" * 50)
print("QUANTITATIVE COMPARISON")
print("=" * 50)
print(f"{'Metric':<30} {'LSTM (NB11)':>15} {'GPT-2 (NB12)':>15}")
print("-" * 62)
print(f"{'Parameters':.<30} {'~3M':>15} {'124M':>15}")
print(f"{'Model size (disk)':.<30} {'~15 MB':>15} {f'{gpt2_size_mb:.0f} MB':>15}")
print(f"{'Vocab size':.<30} {'~5K (word)':>15} {'50K (BPE)':>15}")
print(f"{'Test perplexity':.<30} {'~100 (word)':>15} {f'{perplexity:.1f} (BPE)':>15}")
print(f"{'Coherent beyond 30 tokens':.<30} {'Rarely':>15} {'Often':>15}")
print(f"{'Runs on CPU (fast)':.<30} {'Yes':>15} {'Slow':>15}")
print(f"{'OOV handling':.<30} {'<UNK>':>15} {'Subword split':>15}")
print()
print("NOTE: Perplexity numbers use different tokenizers, so NOT directly comparable.")

In [ ]:
# Inference latency benchmark.
timing_prompt = "This rental offers stunning views of"
n_runs = 5

ft_model.eval()
times = []
for _ in range(n_runs):
    t0 = time.time()
    _ = generate_text(ft_model, timing_prompt, max_new_tokens=40, do_sample=False)
    times.append(time.time() - t0)

avg_time = np.mean(times)
print(f"GPT-2 inference (40 tokens) on {device}:")
print(f"  Mean : {avg_time*1000:.0f} ms")
print(f"  Runs : {[f'{t*1000:.0f}ms' for t in times]}")
print()
# On CPU: expect 2000-5000ms for 40 tokens
# On T4 GPU: expect 100-500ms
# LSTM was typically ~10ms on CPU for the same 40 tokens, i.e. roughly 100-200x faster on CPU.
print("LSTM typically generates 40 tokens in ~10ms on CPU (vs GPT-2's 2000+ ms).")
print("For latency-critical applications, LSTM or distilgpt2 may be better choices.")

## Section 7. Deployment with `pipeline`

In [ ]:
# Production-style inference with pipeline.
gen_pipeline = pipeline(
    'text-generation',
    model=ft_model,
    tokenizer=tokenizer,
    device=0 if device.type == 'cuda' else -1,
)

# Generate multiple samples with a single call
results = gen_pipeline(
    "Bright and airy studio apartment",
    max_new_tokens=GEN_MAX_NEW,
    num_return_sequences=3,
    do_sample=True,
    temperature=GEN_TEMP,
    top_k=GEN_TOP_K,
    top_p=GEN_TOP_P,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)

print("Pipeline output (3 samples):\n")
for i, r in enumerate(results):
    print(f"Sample {i+1}: {r['generated_text'][:300]}\n")

### Optional Lab. Load from Checkpoint and Run Pipeline

In [ ]:
# Solution. Optional Lab: load from checkpoint and run pipeline.

# 1. Load tokenizer + model from SAVE_DIR
# This is the production pattern: you ship SAVE_DIR and recipients call from_pretrained.
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
loaded_model     = AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(device)
loaded_model.eval()

# 2. Build pipeline from the loaded objects
loaded_pipe = pipeline(
    'text-generation',
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    device=0 if device.type == 'cuda' else -1,
)

# 3. Generate 5 samples
outs = loaded_pipe(
    "The host was very helpful and",
    max_new_tokens=40,
    num_return_sequences=5,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=loaded_tokenizer.eos_token_id,
)

print("Generated samples from loaded checkpoint:")
for i, o in enumerate(outs):
    print(f"\nSample {i+1}: {o['generated_text'][:250]}")

# Key insight: reloading from checkpoint produces identical outputs to the in-memory model.
# This proves save_pretrained / from_pretrained is a lossless round-trip.
# In production: store SAVE_DIR in S3/GCS, load once at startup, keep model in memory.
print("\nOutput is Airbnb-style (domain-specific), not generic web text.")
print("Fine-tuning has successfully shifted the model's vocabulary distribution.")

## Section 8. Wrap-up

### When LSTM is still the right answer

- **Edge devices** (Raspberry Pi, mobile): 3M params fit in RAM; 124M may not.
- **Tiny corpora** (<500 samples): fine-tuning GPT-2 on 100 sentences will overfit catastrophically.
- **Real-time low-latency inference on CPU**: an LSTM generates tokens roughly 10-200x faster on CPU.
- **Interpretability**: LSTM hidden states are sometimes easier to probe.

### When GPT-2 wins

- **Open-ended generation**: long, coherent paragraphs where LSTMs drift.
- **Domain shift via fine-tuning**: starting from a 124M-param pretrained checkpoint.
- **BPE tokenization**: handles any word including OOV.
- **Ecosystem**: HuggingFace `pipeline` + `Trainer` + `model.generate()` are industry-standard.

### What's next

The fine-tuning recipe you learned today applies directly to smaller open-source models such as `TinyLlama`, `Phi-2`, and `Gemma-2B`. For **RAG** (pairing generation with vector search), see the `chatbots-with-genai` and `genai_and_llms` courses.

### Self-check quiz (answers)

Q: Why do we set `tokenizer.pad_token = tokenizer.eos_token` for GPT-2?  
A: GPT-2 was trained without a padding token. Batched tokenization needs a consistent pad token ID. We reuse EOS as PAD because (a) it already exists in the vocab and (b) the attention mask ensures EOS-as-PAD positions don't contribute to loss anyway. Without this line you get `ValueError: Asking to pad but the tokenizer does not have a padding token.`

Q: What goes wrong if you set `mlm=True` for GPT-2?  
A: `DataCollatorForLanguageModeling(mlm=True)` randomly masks 15% of tokens and constructs labels only for those positions (the BERT objective). GPT-2 is trained to predict every next token, not just masked ones. With `mlm=True` the model receives corrupted inputs and is asked to reconstruct a subset, which is the wrong objective. Training loss will be erratic and the fine-tuned model will generate incoherent text.

Q: Your fine-tuned GPT-2 test perplexity is higher than the base GPT-2. What are three likely causes?  
A: (a) Learning rate too high (`LR > 5e-4`), so fine-tuning catastrophically forgets pretrained knowledge. (b) No warmup: loss spikes in the first steps and the model never recovers. (c) `mlm=True` instead of `mlm=False`, i.e. wrong objective. Other candidates: overfitting with too many epochs on too little data, or the test set accidentally leaking into training.

Q: Greedy produced `"bedroom bedroom bedroom..."` and top-k=50 didn't help. What else can you try?  
A: Top-k didn't help because the word 'bedroom' is in the top-50 anyway. Try `repetition_penalty=1.2-1.5`, switch to `top_p` sampling, or raise `temperature` above 1.0. The `no_repeat_ngram_size=3` argument also prevents 3-gram repetition: `model.generate(..., no_repeat_ngram_size=3)`.

Q: When is an LSTM-based LM preferable to GPT-2 fine-tuned?  
A: On a Raspberry Pi Zero (512MB RAM) with a 200-sample training corpus and a 50ms latency budget. The LSTM's 3M params fit, train in minutes without a GPU, and generate tokens in microseconds. GPT-2's 124M params don't fit in RAM, require a GPU to fine-tune in reasonable time, and generate tokens 100-200x slower on CPU.

Notebook 12 complete. Notebook 13 is the capstone: Shakespeare generation with LSTM vs. GPT-2.